In [ ]:
from contextlib import asynccontextmanager
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessageChunk,SystemMessage
from fastapi.responses import StreamingResponse
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
import json
import os
import uvicorn

from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is not set")


In [10]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    temperature=0.2,
    max_tokens=4000,
    streaming=True,
)

In [11]:
async def call_model(state: ChatState):
    response = await llm.ainvoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(ChatState)
graph.add_node("model", call_model)
graph.add_edge(START, "model")
graph.add_edge("model", END)


In [12]:
chatbot = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global chatbot
    async with AsyncSqliteSaver.from_conn_string("chatbot.db") as checkpointer:
        chatbot = graph.compile(checkpointer=checkpointer)
        yield

In [13]:

app = FastAPI(lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [14]:
class ChatRequest(BaseModel):
    message: str
    thread_id: str = "default-session"

In [ ]:
def ndjson_event(event_type: str, data: str) -> str:
    return json.dumps({"type": event_type, "data": data}, ensure_ascii=False) + "\n"
#without system mmessage
# async def generate_chat_stream(message: str, thread_id: str):
#     config = {"configurable": {"thread_id": thread_id}}
#     input_message = HumanMessage(content=message)
    
#     try:
#         async for chunk, _metadata in chatbot.astream(
#             {"messages": [input_message]},
#             config=config,
#             stream_mode="messages"
#         ):
#             if isinstance(chunk, AIMessageChunk) and chunk.content:
#                 yield ndjson_event("token", chunk.content)
#     except Exception as e:
#         yield ndjson_event("error", "The model request failed. Please try again.")

async def generate_chat_stream(message: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    
    # 1 Check if history already exists for this thread
    state = await chatbot.aget_state(config)
    existing_messages = (state.values or {}).get("messages", []) if state else []
    
    # 2 If it's a brand new thread (no messages yet), prepend a SystemMessage
    inputs = []
    if not existing_messages:
        inputs.append(SystemMessage(content="You are a helpful, professional AI portfolio assistant. Keep answers clear and concise."))
    
    # 3. Add the user's new message
    inputs.append(HumanMessage(content=message))
    
    try:
        # Pass inputs to LangGraph
        async for chunk, _metadata in chatbot.astream(
            {"messages": inputs},
            config=config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                yield ndjson_event("token", chunk.content)
    except Exception as e:
        yield ndjson_event("error", "The model request failed. Please try again.")

In [ ]:
@app.post("/chat/stream")
async def chat_stream_endpoint(req: ChatRequest):
    return StreamingResponse(
        generate_chat_stream(req.message, req.thread_id),
        media_type="application/x-ndjson"
    )

@app.get("/chat/history/{thread_id}")
async def get_chat_history(thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    state = await chatbot.aget_state(config)
    messages = []
    if state and state.values and "messages" in state.values:
        for msg in state.values["messages"]:
            # Skip system messages so they don't show up on the frontend UI chat boxes
            if msg.type == "system":
                continue
            
            role = "user" if msg.type == "human" else "assistant"
            content = msg.content if isinstance(msg.content, str) else str(msg.content)
            messages.append({"role": role, "content": content})
    return {"messages": messages}


In [17]:
# %%
# Run this inside your Jupyter cell to start the server cleanly without conflict
import uvicorn

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info", reload=False)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [20172]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:63459 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:63459 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:53256 - "GET /chat/history/2724e3e0-3a87-4029-b6f0-ce4dc35cdff0 HTTP/1.1" 200 OK
INFO:     127.0.0.1:53256 - "POST /chat/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:53256 - "POST /chat/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449f-9564-871a6e1905aa HTTP/1.1" 200 OK
INFO:     127.0.0.1:51892 - "GET /chat/history/c17012bb-74e2-449

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [20172]
